# LeetCode 146: LRU Cache

## Problem:
Design a data structure that follows Least Recently Used (LRU) cache constraints.

**Requirements:**
- `get(key)` - Return value if exists, else -1. Mark as recently used.
- `put(key, value)` - Add/update key-value. If capacity exceeded, remove LRU item.
- Both operations must be O(1)

## Approach:
- Use **Doubly Linked List** to maintain order (MRU at head, LRU at tail)
- Use **HashMap** for O(1) key lookup
- When accessed/added, move node to head (most recent)
- When capacity full, remove from tail (least recent)

In [2]:
class Node:
    """Doubly Linked List Node"""
    def __init__(self, key, value):
        self.key = key
        self.value = value
        self.prev = None
        self.next = None


class LRUCache:
    def __init__(self, capacity: int):
        """
        Initialize:
        - capacity
        - cache (hashmap: key -> Node)
        - dummy head and tail nodes (to avoid edge cases)
        """
        self.capacity = capacity
        self.cache = {}  # key -> Node
        
        # Create dummy head and tail
        self.head = Node(0, 0)  # dummy head
        self.tail = Node(0, 0)  # dummy tail
        self.head.next = self.tail
        self.tail.prev = self.head
    
    def _remove(self, node):
        """
        Remove a node from the doubly linked list
        Hint: Update prev and next pointers
        """
        # Disconnect node from list
        node.prev.next = node.next
        node.next.prev = node.prev
    
    def _add_to_head(self, node):
        """
        Add a node right after the dummy head (most recently used position)
        Hint: Insert between head and head.next
        """
        # Insert between dummy head and head.next
        node.next = self.head.next
        node.prev = self.head
        self.head.next.prev = node
        self.head.next = node
    
    def get(self, key: int) -> int:
        """
        Get value by key. If exists, move to head (mark as recently used)
        
        Steps:
        1. Check if key exists in cache
        2. If not, return -1
        3. If yes, remove node from current position
        4. Add node to head (most recent)
        5. Return node value
        """
        if key not in self.cache:
            return -1
        
        node = self.cache[key]
        self._remove(node)
        self._add_to_head(node)
        return node.value
    
    def put(self, key: int, value: int) -> None:
        """
        Add or update key-value pair
        
        Steps:
        1. If key exists:
           - Remove old node
           - Delete from cache
        2. Create new node
        3. Add to cache
        4. Add to head (most recent)
        5. If capacity exceeded:
           - Remove LRU node (before tail)
           - Delete from cache
        """
        # If key exists, remove old node and delete from cache
        if key in self.cache:
            self._remove(self.cache[key])
            del self.cache[key]
        
        # Create new node and add to cache
        new_node = Node(key, value)
        self.cache[key] = new_node
        self._add_to_head(new_node)
        
        # If capacity exceeded, remove LRU
        if len(self.cache) > self.capacity:
            lru_node = self.tail.prev
            self._remove(lru_node)
            del self.cache[lru_node.key]

## Visual Guide:

```
Initial State (capacity = 2):
head <-> tail

After put(1, 1):
head <-> [1:1] <-> tail

After put(2, 2):
head <-> [2:2] <-> [1:1] <-> tail
         (MRU)      (LRU)

After get(1):
head <-> [1:1] <-> [2:2] <-> tail
         (MRU)      (LRU)

After put(3, 3) - evicts key 2:
head <-> [3:3] <-> [1:1] <-> tail
         (MRU)      (LRU)
```

In [3]:
# Test Cases
lRUCache = LRUCache(2)
lRUCache.put(1, 1)  # cache is {1=1}
lRUCache.put(2, 2)  # cache is {1=1, 2=2}
print(lRUCache.get(1))    # return 1
lRUCache.put(3, 3)  # LRU key was 2, evicts key 2, cache is {1=1, 3=3}
print(lRUCache.get(2))    # returns -1 (not found)
lRUCache.put(4, 4)  # LRU key was 1, evicts key 1, cache is {4=4, 3=3}
print(lRUCache.get(1))    # return -1 (not found)
print(lRUCache.get(3))    # return 3
print(lRUCache.get(4))    # return 4

1
-1
-1
3
4


## Expected Output:
```
1
-1
-1
3
4
```